In [1]:
!pip install transformers sentence-transformers faiss-cpu torch sentencepiece -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 75.6 MB/s eta 0:00:00


In [3]:
import torch
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# 1. Create the knowledge base
documents = [
    """
    Generative Artificial Intelligence is a branch of AI that creates
    new content such as text, images, audio, video and computer programs.
    """,
    """
    Large Language Models are transformer-based models trained on massive
    text datasets. They are used for text generation, summarization,
    translation, question answering and conversational AI.
    """,
    """
    Retrieval-Augmented Generation combines information retrieval with
    text generation. It retrieves relevant documents from an external
    knowledge base and gives them to a language model as context.
    """,
    """
    Vector databases store high-dimensional embeddings and perform
    similarity searches. Examples of vector databases include FAISS,
    ChromaDB, Pinecone, Weaviate and Milvus.
    """,
    """
    Prompt engineering is the process of designing clear instructions
    that guide a language model to produce accurate and useful responses.
    Common techniques include zero-shot, few-shot and role-based prompting.
    """,
    """
    Fine-tuning adapts a pretrained language model to a specific domain
    or task by training it further using a smaller domain-specific dataset.
    """
]


# 2. Load the embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


# 3. Convert documents into embeddings
document_embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True
).astype("float32")


# 4. Normalize document embeddings
faiss.normalize_L2(document_embeddings)


# 5. Create the FAISS vector database
embedding_dimension = document_embeddings.shape[1]

# Inner-product search on normalized vectors gives cosine similarity
vector_database = faiss.IndexFlatIP(embedding_dimension)

# Add document embeddings to the database
vector_database.add(document_embeddings)


# 6. Load FLAN-T5 directly
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
generation_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generation_model = generation_model.to(device)
generation_model.eval()

print("Using device:", device)


# 7. Define the retrieval function
def retrieve_documents(query, top_k=2):
    """Retrieve the most relevant documents for a user query."""

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    # Normalize query embedding
    faiss.normalize_L2(query_embedding)

    # Search for similar documents
    similarity_scores, document_indices = vector_database.search(
        query_embedding,
        top_k
    )

    retrieved_documents = []

    for index, score in zip(
        document_indices[0],
        similarity_scores[0]
    ):
        retrieved_documents.append({
            "document": documents[index].strip(),
            "score": float(score)
        })

    return retrieved_documents


# 8. Define the answer-generation function
def generate_answer(query, retrieved_documents):
    """Generate an answer using the retrieved context."""

    context = "\n\n".join(
        item["document"] for item in retrieved_documents
    )

    prompt = f"""
Answer the question using only the information provided in the context.

Context:
{context}

Question:
{query}

Instructions:
1. Give a clear and concise answer.
2. Do not add information that is not present in the context.
3. If the answer is unavailable, state:
   "The answer is not available in the knowledge base."

Answer:
"""

    model_inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = generation_model.generate(
            **model_inputs,
            max_new_tokens=150,
            do_sample=False
        )

    answer = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return answer


# 9. Execute the RAG system
print("\nRETRIEVAL-AUGMENTED GENERATION SYSTEM")
print("=" * 55)

user_query = input("\nEnter your question: ")

retrieved_results = retrieve_documents(
    query=user_query,
    top_k=2
)

answer = generate_answer(
    query=user_query,
    retrieved_documents=retrieved_results
)


# 10. Display retrieved documents
print("\nRETRIEVED DOCUMENTS")
print("-" * 55)

for number, item in enumerate(retrieved_results, start=1):
    print(f"\nDocument {number}:")
    print(item["document"])
    print(f"Similarity Score: {item['score']:.4f}")


# 11. Display generated answer
print("\nGENERATED ANSWER")
print("-" * 55)
print(answer)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Using device: cpu

RETRIEVAL-AUGMENTED GENERATION SYSTEM

Enter your question: What is Retrieval-Augmented Generation?

RETRIEVED DOCUMENTS
-------------------------------------------------------

Document 1:
Retrieval-Augmented Generation combines information retrieval with
    text generation. It retrieves relevant documents from an external
    knowledge base and gives them to a language model as context.
Similarity Score: 0.6933

Document 2:
Generative Artificial Intelligence is a branch of AI that creates
    new content such as text, images, audio, video and computer programs.
Similarity Score: 0.3435

GENERATED ANSWER
-------------------------------------------------------
combines information retrieval with text generation
